# Домашняя бухгалтерия с нейросетью — нейросеть №2 (MLP на TF-IDF)
Блокнот для Google Colab. **Runtime → Change runtime type → T4 GPU**, затем **Run all** (перед ячейкой с реальным тестом нужно будет загрузить файл).

Обучает с нуля (без предобученных весов) небольшую нейросеть: символьные n-граммы TF-IDF → Linear(40 000→128) → ReLU → Dropout → Linear(128→48). Признаки и бейзлайн те же, что в основном классификаторе, поэтому сравнение честное: отличается только скрытый слой.

В конце блокнот скачивает `mlp_outputs.zip` — пришлите его (или содержимое) в чат.

## 1. Код проекта

In [ ]:
import os, subprocess, zipfile, glob
REPO_URL = "https://github.com/osipovajulia274-ops/home-budget.git"   # публичный репозиторий; если он приватный — оставьте "" и загрузите zip проекта в файлы Colab
cwd = os.getcwd()
base = os.path.dirname(cwd) if os.path.basename(cwd) == "home-budget" else cwd
zips = glob.glob(os.path.join(base, "home-budget*.zip"))
if REPO_URL and not os.path.isdir(os.path.join(base, "home-budget")):
    subprocess.run(["git", "clone", REPO_URL, os.path.join(base, "home-budget")], check=True)
elif zips:
    newest = max(zips, key=os.path.getmtime)          # самый свежий загруженный архив
    z = zipfile.ZipFile(newest)
    # архив может быть с папкой home-budget/ наверху или без неё — поддерживаем оба варианта;
    # код перезаписывается, скачанные данные в ml/data не трогаются
    z.extractall(base if all(n.startswith("home-budget/") for n in z.namelist()) else os.path.join(base, "home-budget"))
    print("распакован архив:", os.path.basename(newest))
elif not os.path.isdir(os.path.join(base, "home-budget")):
    raise SystemExit("Укажите REPO_URL или загрузите zip проекта")
os.chdir(os.path.join(base, "home-budget"))
print(os.getcwd(), os.listdir("."))


In [ ]:
!pip -q install pandas scikit-learn pyarrow duckdb pyyaml huggingface_hub pytest matplotlib

## 2. Быстрая проверка кода (без torch-части)

In [ ]:
!python -m pytest -q tests/test_mlp_classifier.py tests/test_categorizer.py

## 3. Данные (как в основном блокноте)
Собирает `category_dataset.parquet` из Open*Facts и словаря: скачивание занимает несколько минут. Если сессия Colab жива и файл уже есть, сборка пропускается.

In [ ]:
import os
REBUILD = False
os.environ['REBUILD'] = '1' if REBUILD else '0'

In [ ]:
%%bash
if [ -f ml/data/processed/category_dataset.parquet ] && [ "$REBUILD" != "1" ]; then
  echo "category_dataset.parquet уже есть - пропускаем сборку"
else
  python -m ml.scripts.01_download && python -m ml.scripts.02_build_category_dataset
fi

### Реальный тест (D6)
Загрузите файл `real_test_labeled.csv` с вашего компьютера (папка `нейросеть\ml\data\processed\`). Он **не** используется при обучении, только для оценки и выбора между моделями.

In [ ]:
import os, shutil
from google.colab import files
target = "ml/data/processed/real_test_labeled.csv"
if not os.path.exists(target):
    up = files.upload()
    for name in up:
        os.makedirs(os.path.dirname(target), exist_ok=True)
        shutil.move(name, target)
        break
print("real_test_labeled.csv:", os.path.exists(target))

## 4. Обучение
Сначала быстрая проверка (2 эпохи на малой выборке), чтобы ошибка не нашлась через 10 минут.

In [ ]:
!python -m ml.train_mlp --smoke

In [ ]:
!python -m ml.train_mlp

## 5. Результаты

In [ ]:
import json
m = json.load(open('ml/reports/metrics_mlp.json', encoding='utf-8'))
print(m['architecture'])
for k in ('test_synth', 'real_test'):
    print('MLP     ', k, {a: round(b, 3) for a, b in m['metrics'].get(k, {}).items() if isinstance(b, float)})
    print('baseline', k, {a: round(b, 3) for a, b in m['baseline'].get(k, {}).items() if isinstance(b, float)})
for name, v in m['metrics'].get('real_test_views', {}).items():
    b = m['baseline']['real_test_views'][name]
    print(f"{name:28s} MLP acc_cat {v['acc_cat']:.3f} F1 {v['macro_f1_cat']:.3f} | baseline acc_cat {b['acc_cat']:.3f} F1 {b['macro_f1_cat']:.3f}")
print('numpy vs torch:', m['numpy_vs_torch'], ' размер, МБ:', m['export']['size_mb'], ' латентность 50 поз., мс:', m['latency_ms_50_items'])
print('выбор:', m.get('production_choice'), '-', m.get('production_choice_reason'))

## 6. Скачать результаты

In [ ]:
import zipfile, os
from google.colab import files
out = 'mlp_outputs.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in ('ml/models_out/mlp_vocab.json', 'ml/models_out/mlp.npz', 'ml/models_out/mlp_choice.json', 'ml/reports/metrics_mlp.json', 'ml/reports/errors_examples_mlp.md'):
        if os.path.exists(p): z.write(p)
print(out, round(os.path.getsize(out) / 1e6, 2), 'МБ')
files.download(out)